# 🤖 Self-RAG Step 7: Query Rewrite — Full Self-RAG!
New this step: if IsUSE says the answer isn't useful, we **rewrite the retrieval query** and try again. This completes the full Self-RAG loop.

## Step 1 — Install

In [ ]:
!pip install -q langchain langchain-community langchain-openai langgraph faiss-cpu pypdf pydantic

## Step 2 — API Key

In [ ]:
import os

# ✏️ Paste your OpenAI API key here
OPENAI_API_KEY = "your-openai-api-key-here"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API Key is set!")

## Step 3 — Imports

In [ ]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END

print("✅ All imports done!")

## Step 4 — Load PDFs

In [ ]:
# Load all 3 company PDFs
docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)
print(f"📄 Loaded {len(docs)} pages total")

## Step 5 — Chunk

In [ ]:
# Split documents into 600-character chunks with 150-character overlap
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
).split_documents(docs)

print(f"✂️  Created {len(chunks)} chunks")

## Step 6 — FAISS

In [ ]:
# Create embeddings and build FAISS vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)

# Retriever returns top 4 most similar chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("✅ FAISS Vector Store is ready!")

## Step 7 — LLM

In [ ]:
# Initialize Gemini Flash — fast and cheap, perfect for RAG
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ LLM ready!")

## Step 8 — State
> ✅ NEW: `retrieval_query` and `rewrite_tries` fields added.

In [ ]:
class State(TypedDict):
    question: str
    retrieval_query: str    # ✅ NEW: rewritten query sent to FAISS
    rewrite_tries: int      # ✅ NEW: how many times we've rewritten
    need_retrieval: bool
    docs: List[Document]
    relevant_docs: List[Document]
    context: str
    answer: str
    issup: Literal["fully_supported", "partially_supported", "no_support"]
    evidence: List[str]
    retries: int
    isuse: Literal["useful", "not_useful"]
    use_reason: str

print("✅ State defined!")

## Step 9 — Decide Retrieval Node

In [ ]:
# Schema for structured LLM output
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(description="True if documents are needed, False if not.")

# Prompt: should we retrieve docs?
decide_retrieval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You decide whether retrieval is needed.\n"
     "Return JSON with key: should_retrieve (boolean).\n\n"
     "Guidelines:\n"
     "- True if answering requires specific facts from company documents.\n"
     "- False for general explanations or definitions.\n"
     "- If unsure, choose True."),
    ("human", "Question: {question}"),
])

should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

# Node 1: Decide if retrieval is needed
def decide_retrieval(state: State):
    decision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}

print("✅ decide_retrieval node ready!")

## Step 10 — Generate Direct Node

In [ ]:
# Prompt: answer directly from general knowledge (no documents)
direct_generation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only your general knowledge.\n"
     "Do NOT assume access to any documents.\n"
     "If unsure, say: 'I don't know based on my general knowledge.'"),
    ("human", "{question}"),
])

# Node 2a: Answer without documents
def generate_direct(state: State):
    out = llm.invoke(direct_generation_prompt.format_messages(question=state["question"]))
    return {"answer": out.content}

print("✅ generate_direct node ready!")

## Step 11 — ✅ UPDATED: Retrieve Node
Now uses `retrieval_query` instead of raw question when available.

In [ ]:
# Updated retrieve: use rewritten query if available
def retrieve(state: State):
    q = state.get("retrieval_query") or state["question"]
    return {"docs": retriever.invoke(q)}

print("✅ retrieve node updated to use retrieval_query!")

## Step 12 — Relevance Grading Node

In [ ]:
# Schema for relevance decision
class RelevanceDecision(BaseModel):
    is_relevant: bool = Field(description="True if the document helps answer the question.")

# Prompt: is this chunk relevant?
is_relevant_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are judging document relevance.\n"
     "A document is relevant if it contains info useful for answering the question."),
    ("human", "Question:\n{question}\n\nDocument:\n{document}"),
])

relevance_llm = llm.with_structured_output(RelevanceDecision)

# Node: Grade each retrieved chunk for relevance
def is_relevant(state: State):
    relevant_docs = []
    for doc in state["docs"]:
        decision = relevance_llm.invoke(
            is_relevant_prompt.format_messages(
                question=state["question"],
                document=doc.page_content
            )
        )
        if decision.is_relevant:
            relevant_docs.append(doc)
    return {"relevant_docs": relevant_docs}

print("✅ is_relevant node ready!")

## Step 13 — Generate from Context + Fallback

In [ ]:
# Prompt: generate answer from retrieved context
rag_generation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a business RAG assistant.\n"
     "Answer the user's question using ONLY the provided context.\n"
     "If the context does not contain enough info, say: 'No relevant document found.'\n"
     "Do not use outside knowledge."),
    ("human", "Question:\n{question}\n\nContext:\n{context}"),
])

# Node: Generate final answer from relevant chunks
def generate_from_context(state: State):
    context = "\n\n---\n\n".join(
        [d.page_content for d in state.get("relevant_docs", [])]
    ).strip()

    if not context:
        return {"answer": "No relevant document found.", "context": ""}

    out = llm.invoke(
        rag_generation_prompt.format_messages(question=state["question"], context=context)
    )
    return {"answer": out.content, "context": context}

# Node: Fallback when no relevant docs found
def no_relevant_docs(state: State):
    return {"answer": "No relevant document found.", "context": ""}

print("✅ generate_from_context node ready!")

## Step 14 — IsSUP + Revise Loop

In [ ]:
# Schema for IsSUP (Is Supported?) check
class IsSUPDecision(BaseModel):
    issup: Literal["fully_supported", "partially_supported", "no_support"]
    evidence: List[str] = Field(default_factory=list)

# Prompt: is the answer grounded in the context?
issup_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are verifying whether the ANSWER is supported by the CONTEXT.\n"
     "Return JSON with keys: issup, evidence.\n"
     "issup must be one of: fully_supported, partially_supported, no_support.\n\n"
     "- fully_supported: Every claim is explicitly in CONTEXT. No interpretive words added.\n"
     "- partially_supported: Core facts are there, BUT answer adds qualitative/interpretive phrasing.\n"
     "- no_support: Key claims are not in CONTEXT.\n\n"
     "Be strict. Evidence = up to 3 short quotes from CONTEXT."),
    ("human", "Question:\n{question}\n\nAnswer:\n{answer}\n\nContext:\n{context}"),
])

issup_llm = llm.with_structured_output(IsSUPDecision)

# Node: Verify answer is grounded in context
def is_sup(state: State):
    decision = issup_llm.invoke(
        issup_prompt.format_messages(
            question=state["question"],
            answer=state.get("answer", ""),
            context=state.get("context", ""),
        )
    )
    return {"issup": decision.issup, "evidence": decision.evidence}

print("✅ is_sup node ready!")

In [ ]:
MAX_RETRIES = 10

# Router: after IsSUP, accept or revise?
def route_after_issup(state: State) -> Literal["accept_answer", "revise_answer"]:
    if state.get("issup") == "fully_supported":
        return "accept_answer"
    if state.get("retries", 0) >= MAX_RETRIES:
        return "accept_answer"   # stop retrying after max attempts
    return "revise_answer"

# Node: Accept the current answer as-is
def accept_answer(state: State):
    return {}  # nothing changes

# Prompt: revise the answer to use ONLY quotes from context
revise_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a STRICT reviser.\n"
     "Output ONLY direct quotes from CONTEXT, formatted as bullet points.\n"
     "FORMAT:\n"
     "- <direct quote from CONTEXT>\n"
     "- <direct quote from CONTEXT>\n\n"
     "Rules:\n"
     "- Use ONLY the CONTEXT. No added words or explanations.\n"
     "- Do NOT say 'context', 'not mentioned', etc."),
    ("human", "Question:\n{question}\n\nCurrent Answer:\n{answer}\n\nCONTEXT:\n{context}"),
])

# Node: Revise the answer to be fully grounded
def revise_answer(state: State):
    out = llm.invoke(
        revise_prompt.format_messages(
            question=state["question"],
            answer=state.get("answer", ""),
            context=state.get("context", ""),
        )
    )
    return {
        "answer":  out.content,
        "retries": state.get("retries", 0) + 1,
    }

print("✅ revise loop nodes ready!")

## Step 15 — IsUSE Node

In [ ]:
# Schema for IsUSE (Is Useful?) check
class IsUSEDecision(BaseModel):
    isuse: Literal["useful", "not_useful"]
    reason: str = Field(description="Short reason in 1 line.")

# Prompt: did the answer actually address the question?
isuse_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are judging USEFULNESS of the ANSWER for the QUESTION.\n"
     "Return JSON with keys: isuse, reason.\n"
     "isuse must be: useful or not_useful.\n\n"
     "- useful: The answer directly answers the question with specific info.\n"
     "- not_useful: The answer is generic, off-topic, or misses the point.\n"
     "Do NOT re-check grounding (IsSUP already did that). Only check: did we answer the question?\n"
     "Keep reason to 1 short line."),
    ("human", "Question:\n{question}\n\nAnswer:\n{answer}"),
])

isuse_llm = llm.with_structured_output(IsUSEDecision)

# Node: Check if the answer is actually useful
def is_use(state: State):
    decision = isuse_llm.invoke(
        isuse_prompt.format_messages(
            question=state["question"],
            answer=state.get("answer", ""),
        )
    )
    return {"isuse": decision.isuse, "use_reason": decision.reason}

# Router: after IsUSE
def route_after_isuse(state: State) -> Literal["END", "no_relevant_docs"]:
    if state.get("isuse") == "useful":
        return "END"
    return "no_relevant_docs"

print("✅ is_use node ready!")

## Step 16 — ✅ NEW: Query Rewrite Node
If the answer isn't useful, rewrite the query and retrieve again.

In [ ]:
MAX_REWRITE_TRIES = 3

# Schema for query rewrite
class RewriteDecision(BaseModel):
    retrieval_query: str = Field(description="Rewritten query optimized for vector retrieval.")

# Prompt: rewrite the question for better FAISS search
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's QUESTION into a short query optimized for vector retrieval over company PDFs.\n"
     "Rules:\n"
     "- Keep it 6–16 words.\n"
     "- Preserve key entities (e.g., NexaAI, plan names).\n"
     "- Add 2–5 high-signal keywords likely in policy/pricing docs.\n"
     "- Remove filler words. Do NOT answer the question.\n"
     "- Output JSON with key: retrieval_query\n\n"
     "Example:\n"
     "Q: 'Do NexaAI plans include a free trial?'\n"
     "→ retrieval_query: 'NexaAI free trial duration period plans'"),
    ("human",
     "QUESTION:\n{question}\n\n"
     "Previous retrieval query:\n{retrieval_query}\n\n"
     "Answer (if any):\n{answer}"),
])

rewrite_llm = llm.with_structured_output(RewriteDecision)

# Node: Rewrite question for better retrieval
def rewrite_question(state: State):
    decision = rewrite_llm.invoke(
        rewrite_prompt.format_messages(
            question=state["question"],
            retrieval_query=state.get("retrieval_query", ""),
            answer=state.get("answer", ""),
        )
    )
    return {
        "retrieval_query": decision.retrieval_query,
        "rewrite_tries":   state.get("rewrite_tries", 0) + 1,
        # Reset retrieval state for a fresh pass
        "docs":            [],
        "relevant_docs":   [],
        "context":         "",
    }

# Updated IsUSE router: if not useful → rewrite → retrieve again
def route_after_isuse(state: State) -> Literal["END", "rewrite_question", "no_relevant_docs"]:
    if state.get("isuse") == "useful":
        return "END"
    if state.get("rewrite_tries", 0) >= MAX_REWRITE_TRIES:
        return "no_relevant_docs"  # give up
    return "rewrite_question"

print("✅ rewrite_question node ready!")

## Step 17 — Routers

In [ ]:
# Router: which path after decide_retrieval?
def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state["need_retrieval"]:
        return "retrieve"
    return "generate_direct"

print("✅ Router ready!")

In [ ]:
# Router: after relevance grading, go to generate or fallback
def route_after_relevance(state: State) -> Literal["generate_from_context", "no_relevant_docs"]:
    if state.get("relevant_docs") and len(state["relevant_docs"]) > 0:
        return "generate_from_context"
    return "no_relevant_docs"

print("✅ Relevance router ready!")

## Step 18 — Build Final Graph
The complete Self-RAG agent with all 7 components wired together.

In [ ]:
g = StateGraph(State)

g.add_node("decide_retrieval",      decide_retrieval)
g.add_node("generate_direct",       generate_direct)
g.add_node("retrieve",              retrieve)
g.add_node("is_relevant",           is_relevant)
g.add_node("generate_from_context", generate_from_context)
g.add_node("no_relevant_docs",      no_relevant_docs)
g.add_node("is_sup",                is_sup)
g.add_node("revise_answer",         revise_answer)
g.add_node("is_use",                is_use)
g.add_node("rewrite_question",      rewrite_question)   # ✅ NEW

g.add_edge(START, "decide_retrieval")
g.add_conditional_edges("decide_retrieval", route_after_decide)
g.add_edge("generate_direct",       END)
g.add_edge("retrieve",              "is_relevant")
g.add_conditional_edges("is_relevant", route_after_relevance)
g.add_edge("no_relevant_docs",      END)
g.add_edge("generate_from_context", "is_sup")
g.add_conditional_edges(
    "is_sup",
    route_after_issup,
    {"accept_answer": "is_use", "revise_answer": "revise_answer"},
)
g.add_edge("revise_answer", "is_sup")

# IsUSE → (END | rewrite → retrieve | give up) ✅ UPDATED
g.add_conditional_edges(
    "is_use",
    route_after_isuse,
    {"END": END, "rewrite_question": "rewrite_question", "no_relevant_docs": "no_relevant_docs"},
)
g.add_edge("rewrite_question", "retrieve")  # ✅ NEW: loop back to retrieve

app = g.compile()
print("✅ Graph compiled!")
print("\nFULL SELF-RAG FLOW:")
print("  decide → retrieve → is_relevant → generate → is_sup ↔ revise → is_use → (END | rewrite → retrieve)")

## Step 19 — Run It! 🚀
The full Self-RAG loop in action.

In [ ]:
question = "Describe NexaAI's company culture."

result = app.invoke(
    {
        "question":        question,
        "retrieval_query": "",   # ✅ starts empty, rewrite fills this
        "rewrite_tries":   0,
        "need_retrieval":  False,
        "docs":            [],
        "relevant_docs":   [],
        "context":         "",
        "answer":          "",
        "issup":           "",
        "evidence":        [],
        "retries":         0,
        "isuse":           "not_useful",
        "use_reason":      "",
    },
    config={"recursion_limit": 80},
)

print("\n===== SELF-RAG RESULT =====")
print(f"❓ Question:      {question}")
print(f"🔍 Need retrieval: {result['need_retrieval']}")
print(f"🔄 Rewrite tries:  {result['rewrite_tries']}")
print(f"🔁 Revise retries: {result['retries']}")
print(f"📚 Relevant docs:  {len(result['relevant_docs'])}")
print(f"🛡️  IsSUP:          {result['issup']}")
print(f"🎯 IsUSE:           {result['isuse']} — {result['use_reason']}")
print(f"\n💬 Final Answer:\n{result['answer']}")
print("=" * 40)